In [ ]:
import os
import json
import hashlib
import shutil
import time
import random
import csv
import threading
from collections import defaultdict

BLOCK_SIZE = 1024 * 8  # 8KB (gerçek HDFS'de 128MB)
REPLICATION = 3

class HDFS:
    def __init__(self, data_dir="/tmp/hdfs_sim"):
        self.data_dir = data_dir
        self.namenode_dir = f"{data_dir}/namenode"
        self.datanode_dir = f"{data_dir}/datanode"
        self.metadata_file = f"{self.namenode_dir}/metadata.json"
        self.lock = threading.Lock()

        os.makedirs(self.namenode_dir, exist_ok=True)
        os.makedirs(self.datanode_dir, exist_ok=True)

        # DataNode listesi (her biri ayrı bir depolama dizini)
        self.datanodes = [f"datanode{i}" for i in range(3)]

        for dn in self.datanodes:
            os.makedirs(f"{self.datanode_dir}/{dn}", exist_ok=True)

        # Metadata yükle
        self.metadata = self._yukle_metadata()

    def _yukle_metadata(self):
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file) as f:
                return json.load(f)
        return {"files": {}}  # dosya_adı -> {blocks, size, replication}

    def _kaydet_metadata(self):
        with open(self.metadata_file, "w") as f:
            json.dump(self.metadata, f, indent=2)

    def _blok_id(self, dosya_adi, blok_no):
        """Her blok için unique ID"""
        return hashlib.md5(f"{dosya_adi}:{blok_no}".encode()).hexdigest()[:12]

    def put(self, kaynak, hedef):
        """Dosyayı HDFS'e yükle (bloklara böl + replication)"""
        basla = time.time()
        dosya_adi = os.path.basename(kaynak)
        hdfs_yolu = f"{hedef}/{dosya_adi}" if hedef != "/" else f"/{dosya_adi}"

        with open(kaynak, "rb") as f:
            data = f.read()

        blok_sayisi = (len(data) + BLOCK_SIZE - 1) // BLOCK_SIZE
        bloklar = []

        for i in range(blok_sayisi):
            blok_data = data[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE]
            blok_id = self._blok_id(hdfs_yolu, i)

            # Her bloku REPLICATION kadar DataNode'a kopyala
            hedef_datanodes = random.sample(self.datanodes, REPLICATION)
            for dn in hedef_datanodes:
                blok_dosyasi = f"{self.datanode_dir}/{dn}/{blok_id}.blk"
                with open(blok_dosyasi, "wb") as bf:
                    bf.write(blok_data)

            bloklar.append({
                "blok_id": blok_id,
                "boyut": len(blok_data),
                "datanodes": hedef_datanodes,
                "blok_no": i
            })

        # Metadata güncelle
        self.metadata["files"][hdfs_yolu] = {
            "size": len(data),
            "blocks": bloklar,
            "replication": REPLICATION,
            "blok_sayisi": blok_sayisi
        }
        self._kaydet_metadata()

        sure = time.time() - basla
        print(f"  ✔ {hdfs_yolu} ({len(data)} bytes) -> {blok_sayisi} blok, {REPLICATION}x replicated [{sure:.3f}s]")

    def get(self, hdfs_yolu, hedef_yerel="."):
        """HDFS'den dosya indir (blokları birleştir, hata tolerant)"""
        basla = time.time()
        meta = self.metadata["files"].get(hdfs_yolu)
        if not meta:
            print(f"  ✗ Dosya bulunamadı: {hdfs_yolu}")
            return

        data = b""
        for blok in meta["blocks"]:
            okundu = False
            for dn in blok["datanodes"]:
                blok_dosyasi = f"{self.datanode_dir}/{dn}/{blok['blok_id']}.blk"
                if os.path.exists(blok_dosyasi):
                    with open(blok_dosyasi, "rb") as bf:
                        data += bf.read()
                    okundu = True
                    break
            if not okundu:
                print(f"  ✗ Blok #{blok['blok_no']} ({blok['blok_id']}) tüm DataNode'larda kayıp!")

        yerel_yol = f"{hedef_yerel}/{os.path.basename(hdfs_yolu)}"
        with open(yerel_yol, "wb") as f:
            f.write(data)

        print(f"  ✔ {hdfs_yolu} -> {yerel_yol} [{time.time()-basla:.3f}s]")

    def ls(self, path="/"):
        """Dosyaları listele"""
        print(f"\n{'DOSYA':<30} {'BOYUT':<10} {'BLOKLAR':<8} {'REPLICATION':<12}")
        print("-" * 60)
        for dosya, meta in sorted(self.metadata["files"].items()):
            if dosya.startswith(path):
                boyut = meta["size"]
                blok = meta["blok_sayisi"]
                rep = meta["replication"]
                print(f"{dosya:<30} {boyut:<10} {blok:<8} {rep:<12}")

    def cat(self, hdfs_yolu):
        """Dosya içeriğini göster (metin)"""
        meta = self.metadata["files"].get(hdfs_yolu)
        if not meta:
            print(f"  ✗ Dosya bulunamadı: {hdfs_yolu}")
            return

        data = b""
        for blok in meta["blocks"]:
            dn = blok["datanodes"][0]
            blok_dosyasi = f"{self.datanode_dir}/{dn}/{blok['blok_id']}.blk"
            with open(blok_dosyasi, "rb") as bf:
                data += bf.read()
        return data.decode()

    def rm(self, hdfs_yolu):
        """Dosya sil"""
        meta = self.metadata["files"].pop(hdfs_yolu, None)
        if not meta:
            print(f"  ✗ Dosya bulunamadı: {hdfs_yolu}")
            return
        for blok in meta["blocks"]:
            for dn in blok["datanodes"]:
                blok_dosyasi = f"{self.datanode_dir}/{dn}/{blok['blok_id']}.blk"
                if os.path.exists(blok_dosyasi):
                    os.remove(blok_dosyasi)
        self._kaydet_metadata()
        print(f"  ✔ {hdfs_yolu} silindi")

    def stat(self, hdfs_yolu):
        """Dosya detaylarını göster"""
        meta = self.metadata["files"].get(hdfs_yolu)
        if not meta:
            print(f"  ✗ Dosya bulunamadı: {hdfs_yolu}")
            return
        print(f"\n📄 {hdfs_yolu}")
        print(f"  Boyut: {meta['size']} bytes ({meta['size']/1024:.1f} KB)")
        print(f"  Blok sayısı: {meta['blok_sayisi']}")
        print(f"  Replication: {meta['replication']}")
        print(f"  Blok detayı:")
        for blok in meta["blocks"]:
            print(f"    #{blok['blok_no']} | ID: {blok['blok_id']} | {blok['boyut']} bytes | DataNode'lar: {blok['datanodes']}")

    # ----- Datanode simülasyonu (node failure) -----
    def datanode_kapat(self, datanode):
        """Bir DataNode'u simüle et (blokları sil)"""
        dn_dizini = f"{self.datanode_dir}/{datanode}"
        if os.path.exists(dn_dizini):
            shutil.rmtree(dn_dizini)
            os.makedirs(dn_dizini)
            print(f"  ⚠ DataNode '{datanode}' kapatıldı (bloklar kayboldu)")

    def datanode_ac(self, datanode):
        """DataNode'u geri aç"""
        os.makedirs(f"{self.datanode_dir}/{datanode}", exist_ok=True)
        print(f"  ✔ DataNode '{datanode}' tekrar açıldı")

    def replication_kontrol(self):
        """Kayıp blokları tespit et ve yeniden replicate et"""
        for dosya, meta in self.metadata["files"].items():
            for blok in meta["blocks"]:
                # Hangi datanode'larda blok gerçekten var?
                var_olan = []
                for dn in blok["datanodes"]:
                    if os.path.exists(f"{self.datanode_dir}/{dn}/{blok['blok_id']}.blk"):
                        var_olan.append(dn)

                if len(var_olan) < REPLICATION:
                    eksik = REPLICATION - len(var_olan)
                    print(f"  ↻ {dosya} blok #{blok['blok_no']}: {eksik} replica eksik")
                    blok["datanodes"] = var_olan  # ölü datanode'ları çıkar
                    if var_olan:
                        kaynak_dn = var_olan[0]
                        kaynak = f"{self.datanode_dir}/{kaynak_dn}/{blok['blok_id']}.blk"
                        if os.path.exists(kaynak):
                            musait_dn = [d for d in self.datanodes if d not in blok["datanodes"]]
                            for yeni_dn in random.sample(musait_dn, min(eksik, len(musait_dn))):
                                shutil.copy2(kaynak, f"{self.datanode_dir}/{yeni_dn}/{blok['blok_id']}.blk")
                                blok["datanodes"].append(yeni_dn)
                                print(f"    → {yeni_dn}'a replica kopyalandı")
        self._kaydet_metadata()


# ============================================================
# KULLANIM
# ============================================================
def data_uret():
    """Örnek CSV verisi üret"""
    urunler = ["Laptop", "Telefon", "Kulaklık", "Mouse", "Klavye", "Monitör", "Tablet"]
    sehirler = ["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya", "Adana"]
    dosya = "/tmp/satislar_ornek.csv"
    with open(dosya, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["tarih", "sehir", "urun", "adet", "fiyat"])
        for _ in range(1000):
            writer.writerow([
                f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
                random.choice(sehirler),
                random.choice(urunler),
                random.randint(1, 10),
                round(random.uniform(100, 5000), 2)
            ])
    return dosya

# MapReduce stili işleme
def map_reduce_satis(hdfs, dosya_yolu):
    """HDFS'deki CSV'yi oku, şehir bazında satış hesapla (MapReduce benzeri)"""
    print("\n--- MapReduce: Şehir bazında toplam satış ---")
    icerik = hdfs.cat(dosya_yolu)
    if not icerik:
        return

    satirlar = icerik.strip().split("\n")[1:]  # header'ı atla

    # MAP: her satır -> (şehir, tutar) çıktısı
    print(f"  MAP aşaması: {len(satirlar)} satır işleniyor...")
    mapped = []
    for satir in satirlar:
        cols = satir.split(",")
        sehir = cols[1]
        tutar = int(cols[3]) * float(cols[4])
        mapped.append((sehir, tutar))

    # SHUFFLE: şehre göre grupla
    print("  SHUFFLE aşaması: gruplama...")
    gruplar = defaultdict(list)
    for sehir, tutar in mapped:
        gruplar[sehir].append(tutar)

    # REDUCE: her şehrin toplamını hesapla
    print("  REDUCE aşaması: toplam hesaplanıyor...")
    sonuc = {}
    for sehir, tutarlar in gruplar.items():
        sonuc[sehir] = sum(tutarlar)

    return dict(sorted(sonuc.items(), key=lambda x: -x[1]))


# ============================================================
if __name__ == "__main__":
    print("=" * 55)
    print("🏗️  HDFS SİMÜLASYONU (Python ile)")
    print("   NameNode + 3 DataNode + Replication")
    print("=" * 55)

    hdfs = HDFS()

    # 1. Veri üret
    print("\n📦 Adım 1: Örnek veri üretiliyor...")
    yerel_dosya = data_uret()
    print(f"   Oluşturulan: {yerel_dosya}")

    # 2. HDFS'e yükle
    print("\n📤 Adım 2: HDFS'e yükle (bloklara böl + replicate)...")
    hdfs.put(yerel_dosya, "/user/data")

    # 3. HDFS durumu
    print("\n📋 Adım 3: HDFS dosya sistemi durumu:")
    hdfs.ls("/user/data")

    # 4. Dosya detayı
    hdfs.stat("/user/data/satislar_ornek.csv")

    # 5. MapReduce işlemi
    print("\n🔍 Adım 4: MapReduce işlemi...")
    sonuc = map_reduce_satis(hdfs, "/user/data/satislar_ornek.csv")

    if sonuc:
        print(f"\n{'ŞEHİR':<15} {'TOPLAM SATIŞ (TL)':<20}")
        print("-" * 35)
        for sehir, tutar in sonuc.items():
            print(f"{sehir:<15} {tutar:>15,.2f}")

    # 6. Node failure + recovery gösterimi
    print("\n\n💥 Adım 5: DataNode hatası ve kurtarma")
    print("   DataNode-0 kapatılıyor (bloklar kayboluyor)...")
    hdfs.datanode_kapat("datanode0")

    print("   Replication kontrol ediliyor...")
    hdfs.replication_kontrol()

    # 7. Dosyayı indir (doğrulama)
    print("\n📥 Adım 6: HDFS'den dosya indir...")
    hdfs.get("/user/data/satislar_ornek.csv", "/tmp")
    print(f"   İndirilen dosya: /tmp/satislar_ornek.csv")

    print("\n" + "=" * 55)
    print("✅ Simülasyon tamamlandı!")
    print("=" * 55)


🏗️  HDFS SİMÜLASYONU (Python ile)
   NameNode + 3 DataNode + Replication

📦 Adım 1: Örnek veri üretiliyor...
   Oluşturulan: /tmp/satislar_ornek.csv

📤 Adım 2: HDFS'e yükle (bloklara böl + replicate)...
  ✔ /user/data/satislar_ornek.csv (36904 bytes) -> 5 blok, 3x replicated [0.001s]

📋 Adım 3: HDFS dosya sistemi durumu:

DOSYA                          BOYUT      BLOKLAR  REPLICATION 
------------------------------------------------------------
/user/data/satislar_ornek.csv  36904      5        3           

📄 /user/data/satislar_ornek.csv
  Boyut: 36904 bytes (36.0 KB)
  Blok sayısı: 5
  Replication: 3
  Blok detayı:
    #0 | ID: 373b4b5591c8 | 8192 bytes | DataNode'lar: ['datanode1', 'datanode0', 'datanode2']
    #1 | ID: 91c9e52a0622 | 8192 bytes | DataNode'lar: ['datanode1', 'datanode2', 'datanode0']
    #2 | ID: c892f5bb6496 | 8192 bytes | DataNode'lar: ['datanode1', 'datanode0', 'datanode2']
    #3 | ID: 7b85559fccf2 | 8192 bytes | DataNode'lar: ['datanode1', 'datanode0', 'datano